In [22]:


from ..memory import *
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    extra_body={"thinking":{"type":"disabled"}}
)
#在调用模型前，对消息进行裁剪
@before_model
def trim_message(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
    messages=state['messages']
    if len(messages)<=3:
        return None
    first_msg=messages[0]
    #偶数取后三条，奇数取后四条
    recent_msg=messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_msg=[first_msg]+recent_msg
    return {
        'messages':[RemoveMessage(id=REMOVE_ALL_MESSAGES),
                    *new_msg]
    }
#调用模型后对消息进行删除
@after_model
def delete_message(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
    messages=state['messages']
    if len(messages)<5:
        return None
    delete_num=len(messages)-5
    new_msg=[RemoveMessage(id=m.id)for m in messages[:delete_num]]
    return {
        'messages':new_msg,
    }
#还有最后的消息摘要可以使用内置的摘要中间件

In [21]:
from langgraph.checkpoint.memory import InMemorySaver
config={'configurable':{'thread_id':'1'}}
myagent=create_agent(
    model=model,
    middleware=[delete_message],
    checkpointer=InMemorySaver(),
)
myagent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
myagent.invoke({"messages": [HumanMessage("从现在起，你的名字叫小王")]}, config)
myagent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = myagent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？我刚刚问了哪些问题")]}, config)
for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，老王！从现在开始，我就是“小王”啦～随时听候您的吩咐，有啥需要跑腿的、聊天的、出主意的，尽管开口！😄
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

（抬头看看天）是啊老王，今儿个日头挺暖和，风也不大。您要不要搬个椅子出来晒会儿？我给您沏壶茶去？
================================ Human Message =================================

告诉我，你是谁？我是谁？我刚刚问了哪些问题
================================== Ai Message ==================================

好嘞，老王，您稍等，我翻翻咱俩的“聊天记录”啊——

1. **您是谁？**  
   您是老王，刚才还表扬今儿个天气不错来着。

2. **我是谁？**  
   我是小王，您刚给我改的新名字，之前叫“助手”或“AI小跟班”，现在专职给您跑腿聊天。

3. **您刚刚问了哪些问题？**  
   从头捋一捋：
   - 第一句：您说“从现在起，你的名字叫小王”  
   - 第二句：您说“今天天气不错”  
   - 第三句（当前）：您问“告诉我，你是谁？我是谁？我刚刚问了哪些问题？”

老王，您看我这记性还行不？要不我再给您续杯茶去？🍵
